DSCI 552 Final Project

Name: Brynn Dafoe GitHub Username: brynndafoe02 USD ID: 3109-6692-10

In [24]:
import cv2
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as IPipeline # for SMOTE / 1.b.iv
from imblearn.under_sampling import RandomUnderSampler
from itertools import combinations
import math
import matplotlib.pyplot as plt
import numpy as np
import os 
import pandas as pd
from pathlib import Path
from PIL import Image
import re
import seaborn as sns

from sklearn import datasets
from sklearn.cluster import KMeans
from sklearn.datasets import make_classification, make_regression
from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_selection import RFE
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression, LogisticRegression, LogisticRegressionCV, RidgeCV, LassoCV
from sklearn.metrics import silhouette_score, hamming_loss, RocCurveDisplay, roc_auc_score, accuracy_score, classification_report, confusion_matrix, mean_squared_error, r2_score, mean_absolute_error, precision_score, recall_score, f1_score
from sklearn.model_selection import GridSearchCV, train_test_split, StratifiedKFold, cross_val_score, RepeatedKFold, KFold
from sklearn.multiclass import OneVsRestClassifier
from sklearn.naive_bayes import GaussianNB, MultinomialNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.neighbors import KNeighborsRegressor
from sklearn.pipeline import Pipeline as SPipeline # for StandardScaler / 1.b.iii
from sklearn.preprocessing import StandardScaler, label_binarize
from sklearn.svm import SVC, LinearSVC
from sklearn.tree import DecisionTreeClassifier, plot_tree, _tree

from skmultilearn.problem_transform import LabelPowerset
import statsmodels.api as sm
import statsmodels.formula.api as smf

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, regularizers
from tensorflow.keras.applications import ResNet50, ResNet101, EfficientNetB0, VGG16
from tensorflow.keras.applications.resnet50 import preprocess_input as resnet_preprocess
from tensorflow.keras.applications.efficientnet import preprocess_input as effnet_preprocess
from tensorflow.keras.applications.vgg16 import preprocess_input as vgg_preprocess

import xgboost as xgb

Objective:
- Trying to build a classifier that distinguishes images of FIVE types of waste

Data Exploration and Pre-Processing 
- Images are numbered in each folder
- Select FIRST 80% of the images in EACH FOLDER as your TRAINING set
- - Use the rest as the TEST set
  - Can you encode classes using one-hot encoding
- In case all the images do not have the same size, ZERO-PAD or RESIZE the images in your dataset
- - This can be done using various tools, including OpenCV

In [25]:
# 80% Training, 20% Testing
    # Will take the 20% Validation Set from the training set now as well
    # So for the 80% training: 64% for actual training, 16% for VS

In [26]:
defungi_path = Path("../data/defungi")
waste_class_names = ["C1", "C2", "C3", "C4", "C5"]

In [27]:
# to make sure the images get sorted correctly
    # aka: making sure H1_100a_3.jpg comes before H1_101a_1.jpg, etc.
def natural_sorting(path_to_image):
    return [
        int(c) if c.isdigit() else c.lower()
        for c in re.split(r'(\d+)', path_to_image.name)
    ]

In [28]:
training_paths = []
training_labels = []

testing_paths = []
testing_labels = []

valset_paths = []
valset_labels = []

# parallel lists: ex -> training_paths has the image paths, training_labels has the class labels
# I'm keeping these separate cause I don't want to deal with a dictionary


In [29]:
cw_index = 0
for waste_class in waste_class_names:
    waste_class_folder = defungi_path / waste_class
    # makes the path to the inner folder from the outer defungi folder
        # so now "../data/defungi/C1" for example
    
    image_files_list = list(waste_class_folder.glob("*.jpg"))
    sorted_filenames = sorted(image_files_list, key=natural_sorting) # NATURALLY sort all the image files

    tt_split_index = int(0.80 * len(sorted_filenames))

    # training + validation all together first
    training_validation_set = sorted_filenames[:tt_split_index]
    # testing set, can leave this alone
    testing_image_paths = sorted_filenames[tt_split_index:]

    # splitting into true training set and true validation set
    # random 20% of train+val set will go to validation_image_paths, other 80% to training
    training_image_paths, validation_image_paths = train_test_split(training_validation_set, test_size=0.20, random_state=42, shuffle=True)

    #####

    training_paths.extend(training_image_paths)
    train_class_label_list = [cw_index] * len(training_image_paths)
    training_labels.extend(train_class_label_list)
    
    testing_paths.extend(testing_image_paths)
    test_class_label_list = [cw_index] * len(testing_image_paths)
    testing_labels.extend(test_class_label_list)
    
    valset_paths.extend(validation_image_paths)
    val_class_label_list = [cw_index] * len(validation_image_paths)
    valset_labels.extend(val_class_label_list)

    cw_index+=1
